<a href="https://colab.research.google.com/github/Sakshambh09/CN-NLP_Lab/blob/main/SakshamBhardwaj_500121198_CN_NLP_Lab_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Experiment-5**

In [1]:
# Cell 1: Install required libraries
!pip install gdown transformers tokenizers sentencepiece spacy nltk pandas
!python -m spacy download en_core_web_sm

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 86.6 MB/s eta 0:00:00
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.


In [2]:
# Cell 2: Download input dataset from Google Drive
import gdown

file_id = "1eHA0Q9ju-08n_hByCK03GqWAX9HQvAKd"
url = f"https://drive.google.com/uc?id={file_id}"
output = "input_text.txt"
gdown.download(url, output, quiet=False)

# Read the input text
with open("input_text.txt", "r", encoding="utf-8") as f:
    input_text = f.read()

# Select a representative test sentence
test_sentence = "Natural language processing combines linguistics, computer science, and machine learning."

Downloading...
From: https://drive.google.com/uc?id=1eHA0Q9ju-08n_hByCK03GqWAX9HQvAKd
To: /content/input_text.txt
100%|██████████| 6.80k/6.80k [00:00<00:00, 12.7MB/s]


**Task 5.1: BPE-Based Subword Tokenization**

**5.1 (a) Using Pretrained Model (GPT-2 BPE Tokenizer)**

In [3]:
# Cell 3: Pretrained BPE Tokenizer
from transformers import GPT2Tokenizer
import pandas as pd

# Load pretrained GPT-2 tokenizer (uses byte-level BPE)
bpe_tokenizer = GPT2Tokenizer.from_pretrained("gpt2")

# Tokenize into subwords and IDs
bpe_tokens = bpe_tokenizer.tokenize(test_sentence)
bpe_ids = bpe_tokenizer.convert_tokens_to_ids(bpe_tokens)

print(f"Original Text: {test_sentence}\n")
df_bpe_pre = pd.DataFrame({"Token": bpe_tokens, "Token ID": bpe_ids})
print(df_bpe_pre.to_string(index=False))

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Original Text: Natural language processing combines linguistics, computer science, and machine learning.

      Token  Token ID
    Natural     35364
  Ġlanguage      3303
Ġprocessing      7587
  Ġcombines     21001
     Ġlingu     20280
     istics      3969
          ,        11
  Ġcomputer      3644
   Ġscience      3783
          ,        11
       Ġand       290
   Ġmachine      4572
  Ġlearning      4673
          .        13


**5.1 (b) Without Using Pretrained Model (Trained from Scratch)**

In [4]:
# Cell 4: Train BPE from scratch using the Tokenizers library
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace
import pandas as pd

# Initialize an empty BPE tokenizer
custom_bpe = Tokenizer(BPE(unk_token="[UNK]"))
custom_bpe.pre_tokenizer = Whitespace()

# Train BPE model directly on the downloaded input dataset
trainer = BpeTrainer(special_tokens=["[UNK]", "[PAD]"], vocab_size=300, min_frequency=1)
custom_bpe.train(["input_text.txt"], trainer)

# Tokenize the test sentence
encoding = custom_bpe.encode(test_sentence)

print(f"Original Text: {test_sentence}\n")
df_bpe_scratch = pd.DataFrame({"Token": encoding.tokens, "Token ID": encoding.ids})
print(df_bpe_scratch.to_string(index=False))

Original Text: Natural language processing combines linguistics, computer science, and machine learning.

     Token  Token ID
         N        15
    atural       183
  language       139
processing       189
       com       104
         b        22
        in        48
        es        54
         l        32
       ing        58
         u        41
     istic       265
         s        39
         ,         2
      comp       176
         u        41
       ter        87
         s        39
         c        23
         i        29
      ence       175
         ,         2
       and        70
        ma       128
     chine       266
  learning       206
         .         4


**Task 5.2: SentencePiece Tokenization**

**5.2 (a) Using Pretrained Model (Albert / T5 SentencePiece)**

In [5]:
# Cell 5: Pretrained SentencePiece Tokenizer
from transformers import AlbertTokenizer
import pandas as pd

# Load pretrained SentencePiece tokenizer
sp_tokenizer = AlbertTokenizer.from_pretrained("albert-base-v2")

# Tokenize into subwords and IDs
sp_tokens = sp_tokenizer.tokenize(test_sentence)
sp_ids = sp_tokenizer.convert_tokens_to_ids(sp_tokens)

print(f"Original Text: {test_sentence}\n")
df_sp_pre = pd.DataFrame({"Token": sp_tokens, "Token ID": sp_ids})
print(df_sp_pre.to_string(index=False))

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/760k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.31M [00:00<?, ?B/s]

Original Text: Natural language processing combines linguistics, computer science, and machine learning.

       Token  Token ID
    ▁natural      1112
   ▁language       816
 ▁processing      5511
   ▁combines     15339
▁linguistics     18165
           ,        15
   ▁computer      1428
    ▁science       762
           ,        15
        ▁and        17
    ▁machine      1940
   ▁learning      2477
           .         9


**5.2 (b) Without Using Pretrained Model (Trained from Scratch)**

In [6]:
# Cell 6: Train SentencePiece model from scratch
import sentencepiece as spm
import pandas as pd

# Train SentencePiece BPE model on the downloaded text
spm.SentencePieceTrainer.train(
    input="input_text.txt",
    model_prefix="custom_spm",
    vocab_size=200,
    model_type="bpe",
    character_coverage=1.0,
    pad_id=0,
    unk_id=1,
    bos_id=2,
    eos_id=3
)

# Load the newly trained SentencePiece model
sp = spm.SentencePieceProcessor(model_file="custom_spm.model")

# Tokenize the test sentence
custom_sp_tokens = sp.encode_as_pieces(test_sentence)
custom_sp_ids = sp.encode_as_ids(test_sentence)

print(f"Original Text: {test_sentence}\n")
df_sp_scratch = pd.DataFrame({"Token": custom_sp_tokens, "Token ID": custom_sp_ids})
print(df_sp_scratch.to_string(index=False))

Original Text: Natural language processing combines linguistics, computer science, and machine learning.

    Token  Token ID
        ▁       154
        N       191
       at        13
     ural       130
▁language       119
     ▁pro        72
  cessing       131
     ▁com       113
        b       172
       in         6
       es        12
       ▁l        25
      ing        17
        u       165
       is        43
      tic       151
        s       161
        ,       173
     ▁com       113
       pu        77
      ter        56
       ▁s        14
        c       164
        i       162
       en         5
       ce       107
        ,       173
     ▁and        46
       ▁m        24
       ac        67
        h       168
       in         6
        e       155
   ▁learn        66
      ing        17
        .       175


**Task 5.3: POS Tagging on Given Sentence**

**Target Sentence: "The young student is reading an interesting book in the library."**

**5.3 (a) Using spaCy**

In [7]:
# Cell 7: POS Tagging with spaCy
import spacy
import pandas as pd

nlp = spacy.load("en_core_web_sm")
sentence_53 = "The young student is reading an interesting book in the library."

doc_53 = nlp(sentence_53)

# Collect unique tokens with POS tag and description
spacy_pos_records = []
seen_tokens = set()

for token in doc_53:
    tok_text = token.text.lower()
    if tok_text not in seen_tokens:
        seen_tokens.add(tok_text)
        spacy_pos_records.append({
            "Unique Token": token.text,
            "POS Tag": token.pos_,
            "Fine-grained Tag": token.tag_,
            "Description": spacy.explain(token.tag_)
        })

df_spacy_pos = pd.DataFrame(spacy_pos_records)
print(df_spacy_pos.to_string(index=False))

Unique Token POS Tag Fine-grained Tag                                        Description
         The     DET               DT                                         determiner
       young     ADJ               JJ adjective (English), other noun-modifier (Chinese)
     student    NOUN               NN                             noun, singular or mass
          is     AUX              VBZ                  verb, 3rd person singular present
     reading    VERB              VBG                 verb, gerund or present participle
          an     DET               DT                                         determiner
 interesting     ADJ               JJ adjective (English), other noun-modifier (Chinese)
        book    NOUN               NN                             noun, singular or mass
          in     ADP               IN          conjunction, subordinating or preposition
     library    NOUN               NN                             noun, singular or mass
           .   PUNCT 

**5.3 (b) Using NLTK**

In [8]:
# Cell 8: POS Tagging with NLTK
import nltk
from nltk import word_tokenize, pos_tag
from nltk.help import upenn_tagset
import pandas as pd

nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('averaged_perceptron_tagger')
nltk.download('averaged_perceptron_tagger_eng')
nltk.download('tagsets')

sentence_53 = "The young student is reading an interesting book in the library."

# Tokenize and run POS tagger
tokens = word_tokenize(sentence_53)
tagged_tokens = pos_tag(tokens)

# Penn Treebank tag description dictionary
nltk_tag_descriptions = {
    "DT": "determiner",
    "JJ": "adjective",
    "NN": "noun, singular or mass",
    "VBZ": "verb, present 3d, 3rd person singular",
    "VBG": "verb, present participle/gerund",
    "IN": "preposition or conjunction, subordinating",
    ".": "punctuation mark, sentence closer"
}

nltk_pos_records = []
seen_tokens = set()

for token, tag in tagged_tokens:
    tok_lower = token.lower()
    if tok_lower not in seen_tokens:
        seen_tokens.add(tok_lower)
        desc = nltk_tag_descriptions.get(tag, tag)
        nltk_pos_records.append({
            "Unique Token": token,
            "POS Tag": tag,
            "Description": desc
        })

df_nltk_pos = pd.DataFrame(nltk_pos_records)
print(df_nltk_pos.to_string(index=False))

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Unzipping taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data] Downloading package tagsets to /root/nltk_data...
[nltk_data]   Unzipping help/tagsets.zip.


Unique Token POS Tag                               Description
         The      DT                                determiner
       young      JJ                                 adjective
     student      NN                    noun, singular or mass
          is     VBZ     verb, present 3d, 3rd person singular
     reading     VBG           verb, present participle/gerund
          an      DT                                determiner
 interesting      JJ                                 adjective
        book      NN                    noun, singular or mass
          in      IN preposition or conjunction, subordinating
     library      NN                    noun, singular or mass
           .       .         punctuation mark, sentence closer


**Task 5.4: POS Tagging on Full Dataset with Frequency (Using spaCy)**

In [9]:
# Cell 9: POS Tagging on input_text.txt with frequencies
import spacy
from collections import Counter
import pandas as pd

nlp = spacy.load("en_core_web_sm")

with open("input_text.txt", "r", encoding="utf-8") as f:
    text_corpus = f.read()

# Process full document with spaCy
doc_full = nlp(text_corpus)

# Extract valid word tokens and their tags (excluding punctuation, newlines, and spaces)
token_tag_pairs = [
    (token.text.lower(), token.pos_, token.tag_)
    for token in doc_full
    if not token.is_punct and not token.is_space
]

# Compute frequencies
frequency_map = Counter(token_tag_pairs)

# Build unique token table
records_54 = []
for (token_text, coarse_pos, fine_tag), freq in frequency_map.items():
    records_54.append({
        "Unique Token": token_text,
        "POS Tag": coarse_pos,
        "Fine Tag": fine_tag,
        "Description": spacy.explain(fine_tag),
        "Frequency": freq
    })

# Format and sort by frequency
df_task_54 = pd.DataFrame(records_54)
df_task_54.sort_values(by="Frequency", ascending=False, inplace=True)

# Save to CSV and show the top 25 records
df_task_54.to_csv("pos_frequency_results.csv", index=False)
print("Saved complete POS tag frequencies to 'pos_frequency_results.csv'.\n")
print(df_task_54.head(25).to_string(index=False))

Saved complete POS tag frequencies to 'pos_frequency_results.csv'.

Unique Token POS Tag Fine Tag                               Description  Frequency
         the     DET       DT                                determiner         47
           a     DET       DT                                determiner         28
         and   CCONJ       CC                 conjunction, coordinating         27
         can     AUX       MD                     verb, modal auxiliary         23
          of     ADP       IN conjunction, subordinating or preposition         22
          be     AUX       VB                           verb, base form         15
          is     AUX      VBZ         verb, 3rd person singular present         14
          as     ADP       IN conjunction, subordinating or preposition         13
         bpe   PROPN      NNP                     noun, proper singular         13
        word    NOUN       NN                    noun, singular or mass         12
    language    NOU